In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.image as mpimg
import math
import re
import yaml

In [2]:
# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)
import plotting
import utils

In [3]:
ROOT_FOLDER = os.path.dirname(os.getcwd())
RUNS_FOLDER = os.path.join(ROOT_FOLDER, "runs")
SINGLE_RUNS_FOLDER = os.path.join(RUNS_FOLDER, "single_runs")
BATCH_RUNS_FOLDER = os.path.join(RUNS_FOLDER, "batch_runs")
DATA_FOLDER = os.path.join(ROOT_FOLDER, "data")
PROCESSED_DATA_FOLDER = os.path.join(DATA_FOLDER, "processed")
CONFIGS_FOLDER = os.path.join(ROOT_FOLDER, "configs")
MODEL_CONFIGS_FOLDER = os.path.join(CONFIGS_FOLDER, "models")

In [4]:
# Load data used for modeling and the results of the model
data_folder_name = "elec_s_37_ES_PT_no_bat_limit"
run_results_folder_name = "Apr09_Wed_h13-GTSEP_v1a_multi-37_ES_PT"

In [5]:
# Paths
run_results_folder = os.path.join(SINGLE_RUNS_FOLDER, run_results_folder_name)
model_info_path = os.path.join(run_results_folder, "model_info")
decision_variables_path = os.path.join(run_results_folder, "decision_variables")
dual_variables_path = os.path.join(run_results_folder, "dual_variables")
model_config_path = os.path.join(model_info_path, "config.yaml")

## Read input data

In [6]:
with open(model_config_path, "r") as file:
    model_config = yaml.safe_load(file)
data_folder_name = model_config["data_folder_name"]
VOLL = model_config["VOLL"]
CC = model_config["CC"]
CO2_price = model_config["CO2_price"]
E_limit = model_config["E_limit"]
expansion_factor = model_config["expansion_factor"]
MS = model_config["MS"]
years = model_config["years"]
years = model_config["years"]
representative_period_unit = model_config["representative_period_unit"]
representative_periods = model_config["representative_periods"]
p_max_new_branch = model_config["p_max_new_branch"]
data_folder = os.path.join(PROCESSED_DATA_FOLDER, data_folder_name)
if representative_period_unit == "week":
    weeks = representative_periods
else:
    print("Error: representative_period_units must be 'week'")

In [7]:
model_config

{'CC': 100,
 'CO2_price': 85,
 'E_limit': inf,
 'MIPGap': 0.01,
 'MS': 0.1,
 'VOLL': 6350,
 'carriers': ['CCGT', 'solar', 'onwind'],
 'clustering_periods': None,
 'clustering_unit': None,
 'data_folder_name': 'elec_s_37_ES_PT_no_bat_limit',
 'discount_rate': 0.07,
 'expansion_factor': 2.0,
 'model_id': '37_ES_PT',
 'model_name': 'GTSEP_v1a_multi',
 'p_max_new_branch': 5000,
 'p_min_new_branch': 100,
 'query': 'not (x > 2 and y < 40)',
 'representative_period_unit': 'week',
 'representative_periods': [21, 42],
 'run_id': 'Apr09_Wed_h13-GTSEP_v1a_multi-37_ES_PT',
 'save_folder': 'C:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\runs\\single_runs\\Apr09_Wed_h13-GTSEP_v1a_multi-37_ES_PT',
 'years': [2025, 2035, 2045]}

In [8]:
input_data = utils.load_multi_year_csv_files_with_week_from_folder(
    years, weeks, data_folder
)
input_data.keys()

dict_keys(['batteries', 'branches', 'capacity_factors', 'generators', 'generator_costs', 'hourly_demand', 'nodes'])

In [9]:
batteries = input_data["batteries"]
branches = input_data["branches"]
capacity_factors = input_data["capacity_factors"]
generators = input_data["generators"]
generator_costs = input_data["generator_costs"]
hourly_demand = input_data["hourly_demand"]
nodes = input_data["nodes"]

In [10]:
generators

bus     carrier         p_nom  marginal_cost  \
year generator                                                          
2025 ES1 0 CCGT        ES1 0        CCGT  26305.332000      40.772380   
     ES1 0 coal        ES1 0        coal   4739.393483      28.196970   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486       0.015000   
     ES1 0 onwind      ES1 0      onwind  26825.862669       0.015000   
     ES1 0 ror         ES1 0         ror    147.008247       0.000000   
     ES1 0 solar       ES1 0       solar  11681.579273       0.010000   
     PT1 0 CCGT        PT1 0        CCGT   4145.000000      38.876697   
     PT1 0 offwind-ac  PT1 0  offwind-ac   4460.188736       0.015000   
     PT1 0 onwind      PT1 0      onwind   5213.640675       0.015000   
     PT1 0 ror         PT1 0         ror   2638.500000       0.000000   
     PT1 0 solar       PT1 0       solar   1025.440000       0.010000   
2035 ES1 0 CCGT        ES1 0        CCGT  26305.332000      40.772380   
     ES1 0 coal        ES1 0        coal   4739.393483      28.196970   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486       0.015000   
     ES1 0 onwind      ES1 0      onwind  26825.862669       0.015000   
     ES1 0 ror         ES1 0         ror    147.008247       0.000000   
     ES1 0 solar       ES1 0       solar  11681.579273       0.010000   
     PT1 0 CCGT        PT1 0        CCGT   4145.000000      38.876697   
     PT1 0 offwind-ac  PT1 0  offwind-ac   4460.188736       0.015000   
     PT1 0 onwind      PT1 0      onwind   5213.640675       0.015000   
     PT1 0 ror         PT1 0         ror   2638.500000       0.000000   
     PT1 0 solar       PT1 0       solar   1025.440000       0.010000   
2045 ES1 0 CCGT        ES1 0        CCGT  26305.332000      40.772380   
     ES1 0 coal        ES1 0        coal   4739.393483      28.196970   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486       0.015000   
     ES1 0 onwind      ES1 0      onwind  26825.862669       0.015000   
     ES1 0 ror         ES1 0         ror    147.008247       0.000000   
     ES1 0 solar       ES1 0       solar  11681.579273       0.010000   
     PT1 0 CCGT        PT1 0        CCGT   4145.000000      38.876697   
     PT1 0 offwind-ac  PT1 0  offwind-ac   4460.188736       0.015000   
     PT1 0 onwind      PT1 0      onwind   5213.640675       0.015000   
     PT1 0 ror         PT1 0         ror   2638.500000       0.000000   
     PT1 0 solar       PT1 0       solar   1025.440000       0.010000   

                        capital_cost  co2_emissions    color  \
year generator                                                 
2025 ES1 0 CCGT         99027.729293           0.20  #b20101   
     ES1 0 coal        349976.553630           0.34  #707070   
     ES1 0 offwind-ac  184160.119456           0.00  #6895dd   
     ES1 0 onwind       96085.888020           0.00  #235ebc   
     ES1 0 ror         299140.224929           0.00  #4adbc8   
     ES1 0 solar        35602.071244           0.00  #f9d002   
     PT1 0 CCGT         99027.729293           0.20  #b20101   
     PT1 0 offwind-ac  185034.009941           0.00  #6895dd   
     PT1 0 onwind       96085.888020           0.00  #235ebc   
     PT1 0 ror         299140.224929           0.00  #4adbc8   
     PT1 0 solar        35602.071244           0.00  #f9d002   
2035 ES1 0 CCGT         99027.729293           0.20  #b20101   
     ES1 0 coal        349976.553630           0.34  #707070   
     ES1 0 offwind-ac  184160.119456           0.00  #6895dd   
     ES1 0 onwind       96085.888020           0.00  #235ebc   
     ES1 0 ror         299140.224929           0.00  #4adbc8   
     ES1 0 solar        35602.071244           0.00  #f9d002   
     PT1 0 CCGT         99027.729293           0.20  #b20101   
     PT1 0 offwind-ac  185034.009941           0.00  #6895dd   
     PT1 0 onwind       96085.888020           0.00  #235ebc   
     PT1 0 ror         299140.224929           0.00  #4adbc8 

In [11]:
branches.index.rename({"line": "branch"}, inplace=True)

In [12]:
decision_variables = utils.load_csv_files_from_folder_multi_weeks(
    decision_variables_path
)
battery_capacity = decision_variables["battery_capacity"]
battery_charging = decision_variables["battery_charging"]
battery_discharging = decision_variables["battery_discharging"]
battery_soc = decision_variables["battery_soc"]
branch_capacity = decision_variables["branch_capacity"]
curtailment = decision_variables["curtailment"]
generation = decision_variables["generation"]
generator_capacity = decision_variables["generator_capacity"]
load_shedding = decision_variables["load_shedding"]
power_flow = decision_variables["power_flow"]

In [13]:
# Update generators with new capacity values with investments
gen_ext = generator_capacity.copy()
gen_ext = gen_ext.reindex(generators.index, fill_value=0)

# Step 2: Add 'extended_by' column
generators["extended_by"] = gen_ext["value"]

# Step 3: Calculate cumulative extension for each generator across years
generators = generators.sort_index()  # sort by year and generator
generators["extended_by_cum"] = generators.groupby("generator")["extended_by"].cumsum()

# Step 4: Add total capacity = p_nom + extended_by_cum
generators["total_capacity"] = generators["p_nom"] + generators["extended_by_cum"]

In [14]:
# Set extenstion potential for branches and generators
branches["extension_potential"] = p_max_new_branch * branches["extendable"]
generators["extension_potential"] = (
    generators["p_nom"] * generators["extendable"] * expansion_factor
)

In [15]:
batteries

node  MC  capital_cost  hour_capacity  cdrate  \
year battery                                                     
2025 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   
2035 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   
2045 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   

                P_discharge_max  P_discharge_min  P_charge_max  P_charge_min  \
year battery                                                                   
2025 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   
2035 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   
2045 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   

                SOC_max  SOC_min     delta  eta_charge  eta_discharge  
year battery                                                           
2025 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97  
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97  
2035 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97  
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97  
2045 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97  
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97

In [16]:
# Step 1: Copy and align battery_capacity with batteries index
battery_ext = battery_capacity.copy()
battery_ext = battery_ext.reindex(batteries.index, fill_value=0)

# Step 2: Add extension for energy capacity (MWh)
batteries["extended_by_soc"] = battery_ext["value"]  # MWh

# Step 3: Add extension for power capacity (MW)
batteries["extended_by_p"] = (
    batteries["extended_by_soc"] / batteries["hour_capacity"]
)  # MW

# Step 4: Cumulative sums over years (grouped by battery ID)
batteries = batteries.sort_index()
batteries["extended_by_soc_cum"] = batteries.groupby("battery")[
    "extended_by_soc"
].cumsum()
batteries["extended_by_p_cum"] = batteries.groupby("battery")["extended_by_p"].cumsum()

# Step 5: Set total capacity fields
batteries["total_energy_capacity"] = batteries["extended_by_soc_cum"]  # MWh
batteries["total_power_capacity"] = batteries["extended_by_p_cum"]  # MW

In [17]:
batteries

node  MC  capital_cost  hour_capacity  cdrate  \
year battery                                                     
2025 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   
2035 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   
2045 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   

                P_discharge_max  P_discharge_min  P_charge_max  P_charge_min  \
year battery                                                                   
2025 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   
2035 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   
2045 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   

                SOC_max  SOC_min     delta  eta_charge  eta_discharge  \
year battery                                                            
2025 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97   
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97   
2035 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97   
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97   
2045 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97   
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97   

                extended_by_soc  extended_by_p  extended_by_soc_cum  \
year battery                                                          
2025 ES1 0 bat         0.000000       0.000000             0.000000   
     PT1 0 bat         0.000000       0.000000             0.000000   
2035 ES1 0 bat         0.000000       0.000000             0.000000   
     PT1 0 bat         0.000000       0.000000             0.000000   
2045 ES1 0 bat     21703.979607   10851.989803         21703.979607   
     PT1 0 bat     24322.849471   12161.424735         24322.849471   

                extended_by_p_cum  total_energy_capacity  total_power_capacity  
year battery                                                                    
2025 ES1 0 bat           0.000000               0.000000              0.000000  
     PT1 0 bat           0.000000               0.000000              0.000000  
2035 ES1 0 bat           0.000000               0.000000              0.000000  
     PT1 0 bat           0.000000               0.000000              0.000000  
2045 ES1 0 bat       10851.989803           21703.979607          10851.989803  
     PT1 0 bat       12161.424735           24322.849471          12161.424735

In [18]:
# Update branches with new capacity values with investments


# Step 1: Ensure 'branches' and 'branch_capacity' are aligned on MultiIndex ["year", "branch"]
branch_ext = branch_capacity.copy()
branch_ext = branch_ext.reindex(branches.index, fill_value=0)

# Step 2: Add 'extended_by' from the decision variable
branches["extended_by"] = branch_ext["value"]

# Step 3: Calculate cumulative extension by branch
branches = branches.sort_index()
branches["extended_by_cum"] = branches.groupby("branch")["extended_by"].cumsum()

# Step 4: Add total capacity = p_max + cumulative additions
branches["total_capacity"] = branches["p_max"] + branches["extended_by_cum"]

In [19]:
generators

bus     carrier         p_nom  marginal_cost  \
year generator                                                          
2025 ES1 0 CCGT        ES1 0        CCGT  26305.332000      40.772380   
     ES1 0 coal        ES1 0        coal   4739.393483      28.196970   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486       0.015000   
     ES1 0 onwind      ES1 0      onwind  26825.862669       0.015000   
     ES1 0 ror         ES1 0         ror    147.008247       0.000000   
     ES1 0 solar       ES1 0       solar  11681.579273       0.010000   
     PT1 0 CCGT        PT1 0        CCGT   4145.000000      38.876697   
     PT1 0 offwind-ac  PT1 0  offwind-ac   4460.188736       0.015000   
     PT1 0 onwind      PT1 0      onwind   5213.640675       0.015000   
     PT1 0 ror         PT1 0         ror   2638.500000       0.000000   
     PT1 0 solar       PT1 0       solar   1025.440000       0.010000   
2035 ES1 0 CCGT        ES1 0        CCGT  26305.332000      40.772380   
     ES1 0 coal        ES1 0        coal   4739.393483      28.196970   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486       0.015000   
     ES1 0 onwind      ES1 0      onwind  26825.862669       0.015000   
     ES1 0 ror         ES1 0         ror    147.008247       0.000000   
     ES1 0 solar       ES1 0       solar  11681.579273       0.010000   
     PT1 0 CCGT        PT1 0        CCGT   4145.000000      38.876697   
     PT1 0 offwind-ac  PT1 0  offwind-ac   4460.188736       0.015000   
     PT1 0 onwind      PT1 0      onwind   5213.640675       0.015000   
     PT1 0 ror         PT1 0         ror   2638.500000       0.000000   
     PT1 0 solar       PT1 0       solar   1025.440000       0.010000   
2045 ES1 0 CCGT        ES1 0        CCGT  26305.332000      40.772380   
     ES1 0 coal        ES1 0        coal   4739.393483      28.196970   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486       0.015000   
     ES1 0 onwind      ES1 0      onwind  26825.862669       0.015000   
     ES1 0 ror         ES1 0         ror    147.008247       0.000000   
     ES1 0 solar       ES1 0       solar  11681.579273       0.010000   
     PT1 0 CCGT        PT1 0        CCGT   4145.000000      38.876697   
     PT1 0 offwind-ac  PT1 0  offwind-ac   4460.188736       0.015000   
     PT1 0 onwind      PT1 0      onwind   5213.640675       0.015000   
     PT1 0 ror         PT1 0         ror   2638.500000       0.000000   
     PT1 0 solar       PT1 0       solar   1025.440000       0.010000   

                        capital_cost  co2_emissions    color  \
year generator                                                 
2025 ES1 0 CCGT         99027.729293           0.20  #b20101   
     ES1 0 coal        349976.553630           0.34  #707070   
     ES1 0 offwind-ac  184160.119456           0.00  #6895dd   
     ES1 0 onwind       96085.888020           0.00  #235ebc   
     ES1 0 ror         299140.224929           0.00  #4adbc8   
     ES1 0 solar        35602.071244           0.00  #f9d002   
     PT1 0 CCGT         99027.729293           0.20  #b20101   
     PT1 0 offwind-ac  185034.009941           0.00  #6895dd   
     PT1 0 onwind       96085.888020           0.00  #235ebc   
     PT1 0 ror         299140.224929           0.00  #4adbc8   
     PT1 0 solar        35602.071244           0.00  #f9d002   
2035 ES1 0 CCGT         99027.729293           0.20  #b20101   
     ES1 0 coal        349976.553630           0.34  #707070   
     ES1 0 offwind-ac  184160.119456           0.00  #6895dd   
     ES1 0 onwind       96085.888020           0.00  #235ebc   
     ES1 0 ror         299140.224929           0.00  #4adbc8   
     ES1 0 solar        35602.071244           0.00  #f9d002   
     PT1 0 CCGT         99027.729293           0.20  #b20101   
     PT1 0 offwind-ac  185034.009941           0.00  #6895dd   
     PT1 0 onwind       96085.888020           0.00  #235ebc   
     PT1 0 ror         299140.224929           0.00  #4adbc8 

In [20]:
def inspect_dataframes_structure(data: dict[str, pd.DataFrame], n: int = 3) -> None:
    """
    Print a structural summary of each dataframe in a dictionary:
    - Name
    - Index names
    - Column names
    - First n rows
    """
    for name, df in data.items():
        print(f"\n📄 DataFrame: {name}")
        print(f"Index names: {df.index.names}")
        print(f"Column names: {list(df.columns)}")
        print("First few rows:")
        print(df.head(n))

In [21]:
# inspect_dataframes_structure(input_data)

In [22]:
dual_variables = utils.load_csv_files_from_folder_multi_weeks(dual_variables_path)

In [23]:
dual_variables.keys()

dict_keys(['battery_charge_new_max_duals', 'battery_charge_old_max_duals', 'battery_charge_old_min_duals', 'battery_discharge_new_max_duals', 'battery_discharge_old_max_duals', 'battery_discharge_old_min_duals', 'branch_extension_duals', 'emissions_dual', 'gen_extension_duals', 'gen_output_new_duals', 'gen_output_old_duals', 'load_shedding_duals', 'power_balance_duals'])

In [24]:
emissions_dual = dual_variables["emissions_dual"]
gen_output_new_duals = dual_variables["gen_output_new_duals"]
gen_output_old_duals = dual_variables["gen_output_old_duals"]
load_shedding_duals = dual_variables["load_shedding_duals"]
power_balance_duals = dual_variables["power_balance_duals"]

In [25]:
inspect_dataframes_structure(dual_variables)


📄 DataFrame: battery_charge_new_max_duals
Index names: ['year', 'week', 'hour', 'battery']
Column names: ['dual_value']
First few rows:
                          dual_value
year week hour battery              
2025 21   0    ES1 0 bat         0.0
          1    ES1 0 bat         0.0
          2    ES1 0 bat         0.0

📄 DataFrame: battery_charge_old_max_duals
Index names: ['year', 'week', 'hour', 'battery']
Column names: ['dual_value']
First few rows:
Empty DataFrame
Columns: [dual_value]
Index: []

📄 DataFrame: battery_charge_old_min_duals
Index names: ['year', 'week', 'hour', 'battery']
Column names: ['dual_value']
First few rows:
Empty DataFrame
Columns: [dual_value]
Index: []

📄 DataFrame: battery_discharge_new_max_duals
Index names: ['year', 'week', 'hour', 'battery']
Column names: ['dual_value']
First few rows:
                          dual_value
year week hour battery              
2025 21   0    ES1 0 bat  -481.31386
          1    ES1 0 bat     0.00000
          2    ES1 0

In [26]:
gen_output_new_duals

dual_value
year week hour generator              
2025 21   0    ES1 0 CCGT     0.000000
          1    ES1 0 CCGT     0.000000
          2    ES1 0 CCGT     0.000000
          3    ES1 0 CCGT     0.000000
          4    ES1 0 CCGT     0.000000
...                                ...
2045 42   163  PT1 0 solar -485.508203
          164  PT1 0 solar -485.508203
          165  PT1 0 solar -485.508203
          166  PT1 0 solar -485.508203
          167  PT1 0 solar -485.508203

[11088 rows x 1 columns]

In [27]:
# Flip sign to get shadow prices (economic value of more generation)
gen_output_new_duals["shadow_price"] = -gen_output_new_duals["dual_value"]

# Filter for binding constraints (positive shadow price)
binding_duals = gen_output_new_duals[gen_output_new_duals["shadow_price"] > 1e-5]

# Group by generator: compute mean, max, and count of binding hours
constrained_stats = (
    binding_duals.groupby("generator")["shadow_price"]
    .agg(["mean", "max", "count"])
    .rename(columns={"count": "binding_hours"})
)

# Add total hours per generator from the full dataset
total_hours = gen_output_new_duals.index.get_level_values("generator").value_counts()
constrained_stats["total_hours"] = constrained_stats.index.map(total_hours)

# Calculate how often each generator is constrained
constrained_stats["binding_ratio"] = (
    constrained_stats["binding_hours"] / constrained_stats["total_hours"]
)

# Sort by mean shadow price (most valuable constraints)
constrained_stats = constrained_stats.sort_values(by="mean", ascending=False)

In [28]:
constrained_stats

,mean,max,binding_hours,total_hours,binding_ratio
generator,,,,,
ES1 0 CCGT,2539.172546,9113.319150,13,1008,0.012897
ES1 0 offwind-ac,531.937641,9615.258289,827,1008,0.820437
ES1 0 onwind,531.937641,9615.258289,827,1008,0.820437
PT1 0 offwind-ac,439.009402,9624.987299,1002,1008,0.994048
PT1 0 onwind,439.009402,9624.987299,1002,1008,0.994048
ES1 0 ror,436.551403,9615.388646,1008,1008,1.000000
PT1 0 ror,436.526608,9625.117656,1008,1008,1.000000
ES1 0 solar,436.464498,9615.301741,1008,1008,1.000000
PT1 0 solar,436.439704,9625.030751,1008,1008,1.000000


In [29]:
high_duals = gen_output_new_duals[gen_output_new_duals["shadow_price"] > 9000].copy()
high_duals = high_duals.reset_index().sort_values(by="shadow_price", ascending=False)
print("High dual entries (shadow_price > 9000):")
high_duals

High dual entries (shadow_price > 9000):


,year,week,hour,generator,dual_value,shadow_price
29,2045,42,91,PT1 0 ror,-9625.117656,9625.117656
28,2045,42,90,PT1 0 ror,-9625.117656,9625.117656
27,2045,42,89,PT1 0 ror,-9625.117656,9625.117656
32,2045,42,91,PT1 0 solar,-9625.030751,9625.030751
31,2045,42,90,PT1 0 solar,-9625.030751,9625.030751
30,2045,42,89,PT1 0 solar,-9625.030751,9625.030751
26,2045,42,91,PT1 0 onwind,-9624.987299,9624.987299
25,2045,42,90,PT1 0 onwind,-9624.987299,9624.987299
24,2045,42,89,PT1 0 onwind,-9624.987299,9624.987299
23,2045,42,91,PT1 0 offwind-ac,-9624.987299,9624.987299


In [30]:
merged = high_duals.merge(
    load_shedding.reset_index(),
    on=["year", "week", "hour"],
    how="left",
    suffixes=("", "_load_shed"),
)

# See if any load was shed at these high-dual hours
print("\nLoad shedding in high-dual hours:")
print(merged[merged["value"] > 1e-3].sort_values("shadow_price", ascending=False))


Load shedding in high-dual hours:
Empty DataFrame
Columns: [year, week, hour, generator, dual_value, shadow_price, node, value]
Index: []


In [31]:
# Merge generation with capacity factors
dispatch = generation.reset_index()
cf = capacity_factors.reset_index()
dispatch = dispatch.merge(cf, on=["year", "week", "hour"])

# Get the installed capacity (p_nom) from the generators dataframe
dispatch["p_nom"] = dispatch.apply(
    lambda row: (
        generators.loc[(row["year"], row["generator"]), "p_nom"]
        if (row["year"], row["generator"]) in generators.index
        else None
    ),
    axis=1,
)

# Get the correct column name from capacity_factors: it's dynamic, e.g. "PT1 0 solar"
# Use the row's generator as a column name to extract the capacity factor
dispatch["cf"] = dispatch.apply(lambda row: row.get(row["generator"], None), axis=1)

# Calculate the theoretical output cap
dispatch["dispatch_limit"] = dispatch["p_nom"] * dispatch["cf"]

# Calculate utilization ratio
dispatch["utilization"] = dispatch["value"] / dispatch["dispatch_limit"]

In [32]:
# Get just year, week, hour combinations of interest
high_hours = high_duals[["year", "week", "hour"]].drop_duplicates()

# Merge to get dispatch in those hours only
dispatch_high = dispatch.merge(high_hours, on=["year", "week", "hour"])

# Look at generators near or at their cap
near_limit = dispatch_high[dispatch_high["utilization"] >= 0.95]

# View summary
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("\nGenerators near their limit during high-dual hours:")
    print(
        near_limit[
            [
                "year",
                "week",
                "hour",
                "generator",
                "value",
                "dispatch_limit",
                "utilization",
            ]
        ]
    )


Generators near their limit during high-dual hours:
    year  week  hour         generator         value  dispatch_limit  \
0   2045    42    89        ES1 0 CCGT  58476.920303    26305.332000   
1   2045    42    90        ES1 0 CCGT  58476.920303    26305.332000   
2   2045    42    91        ES1 0 CCGT  58476.920303    26305.332000   
3   2045    42    89        ES1 0 coal   4739.393483     4739.393483   
4   2045    42    90        ES1 0 coal   4739.393483     4739.393483   
5   2045    42    91        ES1 0 coal   4739.393483     4739.393483   
6   2045    42    89  ES1 0 offwind-ac   3065.598445     1021.866148   
7   2045    42    90  ES1 0 offwind-ac   2599.743938      866.581313   
8   2045    42    91  ES1 0 offwind-ac   2522.652764      840.884255   
9   2045    42    89      ES1 0 onwind   5404.025773     1080.805155   
10  2045    42    90      ES1 0 onwind   6033.940878     1206.788176   
11  2045    42    91      ES1 0 onwind   6331.169404     1266.233881   
12  2045   

In [33]:
# Extract and drop duplicates
near_limit_timesteps = near_limit[["year", "week", "hour"]].drop_duplicates()

# Reset index for convenience
near_limit_timesteps = near_limit_timesteps.reset_index(drop=True)

# Display result
print("Timesteps when at least one generator was near its limit:")
print(near_limit_timesteps)

Timesteps when at least one generator was near its limit:
   year  week  hour
0  2045    42    89
1  2045    42    90
2  2045    42    91


In [34]:
key_times = set(
    [tuple(x) for x in near_limit_timesteps[["year", "week", "hour"]].values]
)
key_times

{(2045, 42, 89), (2045, 42, 90), (2045, 42, 91)}

In [35]:
generators.keys()

Index(['bus', 'carrier', 'p_nom', 'marginal_cost', 'capital_cost',
       'co2_emissions', 'color', 'nice_name', 'extendable',
       'extension_potential', 'extended_by', 'extended_by_cum',
       'total_capacity'],
      dtype='object')

In [36]:
# Filter generation
gen_filtered = generation.reset_index()
gen_filtered = gen_filtered[
    gen_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(key_times)
]

# Filter capacity factors
cf_filtered = capacity_factors.reset_index()
cf_filtered = cf_filtered[
    cf_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(key_times)
]

# Step 2: Merge generation with capacity factors
gen_cf = gen_filtered.merge(cf_filtered, on=["year", "week", "hour"])

# Step 3: Add total_capacity from generators
gen_cf["total_capacity"] = gen_cf.apply(
    lambda row: (
        generators.loc[(row["year"], row["generator"]), "total_capacity"]
        if (row["year"], row["generator"]) in generators.index
        else None
    ),
    axis=1,
)

# Step 4: Add capacity factor (the column name is the generator name)
gen_cf["cf"] = gen_cf.apply(lambda row: row.get(row["generator"], None), axis=1)

# Step 5: Compute dispatch limit
gen_cf["dispatch_limit"] = gen_cf["total_capacity"] * gen_cf["cf"]

# Step 6: Output generation vs dispatch limit
gen_cf["utilization_ratio"] = gen_cf["value"] / gen_cf["dispatch_limit"]

# Optional: clean up and sort
result = gen_cf[
    [
        "year",
        "week",
        "hour",
        "generator",
        "value",
        "dispatch_limit",
        "utilization_ratio",
    ]
]
result = result.sort_values(by=["year", "week", "hour", "generator"]).reset_index(
    drop=True
)

# Display the final result
print("Generation vs theoretical maximum at near-limit timesteps:")
print(result)

Generation vs theoretical maximum at near-limit timesteps:
    year  week  hour         generator         value  dispatch_limit  \
0   2045    42    89        ES1 0 CCGT  58476.920303    58476.920303   
1   2045    42    89        ES1 0 coal   4739.393483     4739.393483   
2   2045    42    89  ES1 0 offwind-ac   3065.598445     3065.598445   
3   2045    42    89      ES1 0 onwind   5404.025773     5404.025773   
4   2045    42    89         ES1 0 ror      7.234849        7.234849   
5   2045    42    89       ES1 0 solar    427.942386      427.942386   
6   2045    42    89        PT1 0 CCGT  29015.000000    29015.000000   
7   2045    42    89  PT1 0 offwind-ac    164.629173      164.629173   
8   2045    42    89      PT1 0 onwind     39.962327       39.962327   
9   2045    42    89         PT1 0 ror    140.743869      140.743869   
10  2045    42    89       PT1 0 solar    206.022422      206.022422   
11  2045    42    90        ES1 0 CCGT  58476.920303    58476.920303   
12  2

In [ ]:
branches[["capital_cost"]]

,,capital_cost
year,branch,
2025,35,26476.107009
2035,35,26476.107009
2045,35,26476.107009


In [ ]:
# Merge generation duals with load shedding to compare
gen_duals = gen_output_new_duals["dual_value"].reset_index()
gen_duals["shadow_price"] = -gen_duals["dual_value"]

load_shed = load_shedding["value"].reset_index()
merged = pd.merge(gen_duals, load_shed, on=["year", "week", "hour"], how="left")

# Only keep rows where shadow price is near VOLL
threshold = 0.95 * VOLL
high_dual_hours = merged[merged["shadow_price"] >= threshold]

# Display some high-dual entries with load shedding info
print("\nHigh-dual hours and associated load shedding:")
with pd.option_context("display.max_rows", None):
    print(high_dual_hours.sort_values(by="shadow_price", ascending=False))


High-dual hours and associated load shedding:
       year  week  hour         generator   dual_value  shadow_price   node  \
20005  2045    42    90         PT1 0 ror -9625.117656   9625.117656  PT1 0   
20007  2045    42    91         PT1 0 ror -9625.117656   9625.117656  PT1 0   
20002  2045    42    89         PT1 0 ror -9625.117656   9625.117656  ES1 0   
20003  2045    42    89         PT1 0 ror -9625.117656   9625.117656  PT1 0   
20004  2045    42    90         PT1 0 ror -9625.117656   9625.117656  ES1 0   
20006  2045    42    91         PT1 0 ror -9625.117656   9625.117656  ES1 0   
22023  2045    42    91       PT1 0 solar -9625.030751   9625.030751  PT1 0   
22018  2045    42    89       PT1 0 solar -9625.030751   9625.030751  ES1 0   
22019  2045    42    89       PT1 0 solar -9625.030751   9625.030751  PT1 0   
22020  2045    42    90       PT1 0 solar -9625.030751   9625.030751  ES1 0   
22021  2045    42    90       PT1 0 solar -9625.030751   9625.030751  PT1 0   
22022

In [51]:
print(key_times)
print(prev_key_times)

{(2045, 42, 89), (2045, 42, 90), (2045, 42, 91)}
{(2045, 42, 89), (2045, 42, 90), (2045, 42, 88)}


In [47]:
soc_current

,year,week,hour,battery,value
929,2045,42,89,ES1 0 bat,16427.608274
930,2045,42,90,ES1 0 bat,7105.039000
931,2045,42,91,ES1 0 bat,2170.397961
1937,2045,42,89,PT1 0 bat,16252.432182
1938,2045,42,90,PT1 0 bat,8793.420026
1939,2045,42,91,PT1 0 bat,2432.284947


In [48]:
soc_prev

,year,week,hour,battery,value_soc_prev
929,2045,42,88,ES1 0 bat,16427.608274
930,2045,42,89,ES1 0 bat,7105.039000
931,2045,42,90,ES1 0 bat,2170.397961
1937,2045,42,88,PT1 0 bat,16252.432182
1938,2045,42,89,PT1 0 bat,8793.420026
1939,2045,42,90,PT1 0 bat,2432.284947


In [54]:
combined_set = key_times.union(prev_key_times)
print(
    soc_filtered[
        soc_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(combined_set)
    ]
)

      year  week  hour    battery         value
928   2045    42    88  ES1 0 bat  19533.581646
929   2045    42    89  ES1 0 bat  16427.608274
930   2045    42    90  ES1 0 bat   7105.039000
931   2045    42    91  ES1 0 bat   2170.397961
1936  2045    42    88  PT1 0 bat  21890.564524
1937  2045    42    89  PT1 0 bat  16252.432182
1938  2045    42    90  PT1 0 bat   8793.420026
1939  2045    42    91  PT1 0 bat   2432.284947


In [ ]:
# Step 1: Prepare key times from near_limit_timesteps
key_times = set(tuple(x) for x in near_limit_timesteps[["year", "week", "hour"]].values)

# Step 2: Filter battery SOC and discharge for relevant timesteps
discharge_filtered = battery_discharging.reset_index()
soc_filtered = battery_soc.reset_index()

# Filter for target hours
discharge_filtered = discharge_filtered[
    discharge_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(key_times)
]
soc_current = soc_filtered[
    soc_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(key_times)
]
prev_key_times = {(year, week, hour - 1) for year, week, hour in key_times}
# Step 3: Compute previous time steps
soc_prev = (
    soc_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(prev_key_times)
)


# Rename column for clarity
soc_prev = soc_prev.rename(columns={"value": "value_soc_prev"})

# Step 4: Merge current SOC with previous SOC
soc_merged = soc_current.merge(
    soc_prev[["year", "week", "hour", "battery", "value_soc_prev"]],
    on=["year", "week", "hour", "battery"],
    how="left",
)

# Step 5: Merge with discharge
battery_status = discharge_filtered.merge(
    soc_merged,
    on=["year", "week", "hour", "battery"],
    how="inner",
    suffixes=("_discharge", "_soc"),
)

# Step 6: Add cumulative capacity info from batteries
battery_status["SOC_max_frac"] = battery_status.apply(
    lambda row: (
        batteries.loc[(row["year"], row["battery"]), "SOC_max"]
        if (row["year"], row["battery"]) in batteries.index
        else None
    ),
    axis=1,
)
battery_status["total_energy_capacity"] = battery_status.apply(
    lambda row: (
        batteries.loc[(row["year"], row["battery"]), "total_energy_capacity"]
        if (row["year"], row["battery"]) in batteries.index
        else None
    ),
    axis=1,
)
battery_status["total_power_capacity"] = battery_status.apply(
    lambda row: (
        batteries.loc[(row["year"], row["battery"]), "total_power_capacity"]
        if (row["year"], row["battery"]) in batteries.index
        else None
    ),
    axis=1,
)

# Step 7: Compute max SOC and ratios
battery_status["SOC_max"] = (
    battery_status["SOC_max_frac"] * battery_status["total_energy_capacity"]
)
battery_status["soc_ratio"] = battery_status["value_soc"] / battery_status["SOC_max"]
battery_status["discharge_ratio"] = (
    battery_status["value_discharge"] / battery_status["total_power_capacity"]
)
battery_status["soc_ratio_prev"] = (
    battery_status["value_soc_prev"] / battery_status["SOC_max"]
)

# Step 8: Final cleaned-up output
battery_usage_summary = (
    battery_status[
        [
            "year",
            "week",
            "hour",
            "battery",
            "value_discharge",
            "total_power_capacity",
            "discharge_ratio",
            "value_soc_prev",
            "value_soc",
            "SOC_max",
            "soc_ratio_prev",
            "soc_ratio",
        ]
    ]
    .sort_values(by=["year", "week", "hour", "battery"])
    .reset_index(drop=True)
)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Battery usage summary:")
    print(battery_usage_summary)

TypeError: Series.rename() got an unexpected keyword argument 'columns'

In [ ]:
# Step 1: Get the key (year, week, hour) tuples
key_times = set(
    [tuple(x) for x in near_limit_timesteps[["year", "week", "hour"]].values]
)

# Step 2: Filter power_flow for those timesteps
flow_filtered = power_flow.reset_index()
flow_filtered = flow_filtered[
    flow_filtered[["year", "week", "hour"]].apply(tuple, axis=1).isin(key_times)
]

# Step 3: Add total capacity (from branches dataframe)
flow_filtered["total_capacity"] = flow_filtered.apply(
    lambda row: (
        branches.loc[(row["year"], row["branch"]), "total_capacity"]
        if (row["year"], row["branch"]) in branches.index
        else None
    ),
    axis=1,
)

# Step 4: Calculate flow utilization (abs value / capacity)
flow_filtered["abs_flow"] = flow_filtered["value"].abs()
flow_filtered["flow_ratio"] = (
    flow_filtered["abs_flow"] / flow_filtered["total_capacity"]
)

# Step 5: Clean output
flow_summary = (
    flow_filtered[
        ["year", "week", "hour", "branch", "value", "total_capacity", "flow_ratio"]
    ]
    .sort_values(by=["year", "week", "hour", "branch"])
    .reset_index(drop=True)
)

print("Power flow vs max capacity at near-limit timesteps:")
print(flow_summary)

Power flow vs max capacity at near-limit timesteps:
   year  week  hour  branch        value  total_capacity  flow_ratio
0  2045    42    89      35 -16297.22093     16297.22093         1.0
1  2045    42    90      35 -16297.22093     16297.22093         1.0
2  2045    42    91      35 -16297.22093     16297.22093         1.0


In [ ]:
input_data.keys()

dict_keys(['batteries', 'branches', 'capacity_factors', 'generators', 'generator_costs', 'hourly_demand', 'nodes'])

In [ ]:
decision_variables.keys()

dict_keys(['battery_capacity', 'battery_charging', 'battery_discharging', 'battery_soc', 'branch_capacity', 'curtailment', 'generation', 'generator_capacity', 'load_shedding', 'power_flow'])

In [ ]:
dual_variables.keys()

dict_keys(['battery_charge_new_max_duals', 'battery_charge_old_max_duals', 'battery_charge_old_min_duals', 'battery_discharge_new_max_duals', 'battery_discharge_old_max_duals', 'battery_discharge_old_min_duals', 'branch_extension_duals', 'emissions_dual', 'gen_extension_duals', 'gen_output_new_duals', 'gen_output_old_duals', 'load_shedding_duals', 'power_balance_duals'])